# 06 — Robustness Checks

This notebook addresses **Key Outstanding Questions §2** from the README.

## Checks implemented

| Check | Method | Status |
|-------|--------|--------|
| **θ-sensitivity** | Repeat core metrics for $\theta_{\max} \in \{20°, 30°, 40°\}$ | Requires per-θ caches |
| **Bench inclusion** | Compare active-only vs all-players centroid | ✓ |
| **Cross-team** | Repeat per-team and compare effect sizes | ✓ |
| **Speed confounding** | Partial correlation of order controlling for speed | ✓ |
| **Selection effect** | Hazard conditioned on survival to age 10s, split by start order | ✓ |

**Prerequisite**: Run `01_data_loading.ipynb` first.

In [ ]:
import sys, os
from pathlib import Path

_HERE = Path(os.getcwd())
_REPO = _HERE.parents[2]
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

from analysis.levy_paper.util.paper_utils import (
    configure_paper_plotting,
    load_cache, save_cache, save_figure,
    plot_ccdf,
    THETA_GRID, STATE_COLORS, STATE_LABELS, DATA_DIR,
)
configure_paper_plotting()

centroid_runs = load_cache("centroid_order_runs")
runs_long     = load_cache("runs_long")
print(f"Base runs: {len(runs_long):,}")
print(f"Centroid runs: {len(centroid_runs):,}")

## 1. Turning-Angle Sensitivity ($\theta_{\max} \in \{20°, 30°, 40°\}$)

Does the heavy-tail signature and the order–transport coupling hold across
different run segmentation thresholds?

**How to generate caches**: Re-run `01_data_loading.ipynb` with `THETA_DEG`
overridden to 20 or 40, saving outputs as `centroid_order_runs_theta20` etc.

In [ ]:
theta_results = {}

for theta in THETA_GRID:
    cache_name = f"centroid_order_runs_theta{theta}"
    try:
        theta_results[theta] = load_cache(cache_name)
        print(f"  theta={theta}: loaded from cache ({len(theta_results[theta]):,} runs)")
    except FileNotFoundError:
        if theta == 30:
            # Base case: use the default cache
            theta_results[theta] = centroid_runs.copy()
            print(f"  theta={theta}: using default cache")
        else:
            print(f"  theta={theta}: cache not found — re-run 01_data_loading.ipynb with THETA_DEG={theta}")

In [ ]:
theta_colors = {20: "#4393c3", 30: "#1b7837", 40: "#d6604d"}
theta_ls     = {20: "--",      30: "-",       40: ":"}

fig_theta, axes_theta = plt.subplots(1, 2, figsize=(9, 4))

for theta, df in theta_results.items():
    label = f"theta={theta}" + (" (base)" if theta == 30 else "")
    plot_ccdf(axes_theta[0], df["duration_s"],
              label=label, color=theta_colors[theta], ls=theta_ls[theta])
    plot_ccdf(axes_theta[1], df["run_length_m"],
              label=label, color=theta_colors[theta], ls=theta_ls[theta])

axes_theta[0].set_xlabel("Duration (s)"); axes_theta[0].set_ylabel(r"$P(T \geq t)$")
axes_theta[0].set_title("Duration CCDF by theta")
axes_theta[0].legend(fontsize=8)

axes_theta[1].set_xlabel("Length (m)"); axes_theta[1].set_ylabel(r"$P(L \geq l)$")
axes_theta[1].set_title("Length CCDF by theta")
axes_theta[1].legend(fontsize=8)

plt.tight_layout()
save_figure(fig_theta, "supp_robustness_theta_sensitivity")
plt.show()

# Summary stats per theta
rows = []
for theta, df in theta_results.items():
    rows.append({
        "theta": theta,
        "n_runs": len(df),
        "median_duration_s": df["duration_s"].median(),
        "median_length_m": df["run_length_m"].median(),
        "mean_speed_mps": df["v_mean_mps"].mean(),
    })
print(pd.DataFrame(rows).set_index("theta").round(2))

## 2. Bench-Player Inclusion

Does including substitutes / bench players substantially change the centroid?

We check what fraction of frames have fewer than 11 active players, and whether
the centroid statistics change meaningfully when those frames are excluded.

In [ ]:
traj = load_cache("trajectory_long")

# Centroid rows only (track_type == 'centroid')
centroid_traj = traj.loc[traj["track_type"] == "centroid"]

if "n_active" in centroid_traj.columns:
    n_active_vals = centroid_traj["n_active"].dropna()

    print("Active player count distribution:")
    print(n_active_vals.value_counts().sort_index())
    frac_lt11 = (n_active_vals < 11).mean()
    print(f"\nFraction of centroid frames with < 11 active players: {frac_lt11:.1%}")

    if frac_lt11 > 0.05:
        print("WARNING: >5% of frames have fewer than 11 players — bench exclusion may matter.")
    else:
        print("OK: bench exclusion has minimal impact (<5% of frames affected).")

    fig_bench, ax_bench = plt.subplots(figsize=(5, 3))
    n_active_vals.hist(bins=range(int(n_active_vals.min()), int(n_active_vals.max()) + 2),
                       ax=ax_bench, edgecolor="k", lw=0.3, color="#4393c3")
    ax_bench.axvline(11, color="red", ls="--", lw=1, label="Full squad (11)")
    ax_bench.set_xlabel("Active players per frame")
    ax_bench.set_ylabel("Count")
    ax_bench.set_title("Active player count distribution")
    ax_bench.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("'n_active' column not in trajectory_long — bench exclusion check skipped.")
    print(f"trajectory_long has {len(centroid_traj):,} centroid rows across "
          f"{centroid_traj['match_id'].nunique()} matches.")

## 3. Cross-Team Consistency

Do the core patterns hold across different teams?

Plot CCDFs per team and compute the order–duration correlation for each.

In [ ]:
if "team" in centroid_runs.columns:
    teams = sorted(centroid_runs["team"].unique())
    team_palette = ["#1b7837", "#762a83", "#d6604d", "#4393c3",
                    "#f4a582", "#92c5de", "#a6dba0"]

    fig_teams, axes_t = plt.subplots(1, 2, figsize=(9, 4))

    team_stats = []
    for i, team in enumerate(teams):
        sub = centroid_runs.loc[centroid_runs["team"] == team]
        c = team_palette[i % len(team_palette)]
        plot_ccdf(axes_t[0], sub["duration_s"],    label=str(team), color=c)
        plot_ccdf(axes_t[1], sub["run_length_m"],  label=str(team), color=c)

        r_dur = sub["p_mean"].corr(sub["duration_s"])   if "p_mean" in sub.columns else np.nan
        r_len = sub["p_mean"].corr(sub["run_length_m"]) if "p_mean" in sub.columns else np.nan
        team_stats.append({"team": team, "n_runs": len(sub),
                           "r_order_duration": r_dur, "r_order_length": r_len,
                           "median_duration_s": sub["duration_s"].median()})

    axes_t[0].set_xlabel("Duration (s)"); axes_t[0].set_ylabel(r"$P(T \geq t)$")
    axes_t[0].set_title("Duration CCDF per team"); axes_t[0].legend(fontsize=8)
    axes_t[1].set_xlabel("Length (m)"); axes_t[1].set_ylabel(r"$P(L \geq l)$")
    axes_t[1].set_title("Length CCDF per team"); axes_t[1].legend(fontsize=8)

    plt.tight_layout()
    save_figure(fig_teams, "supp_robustness_cross_team")
    plt.show()

    print("\nPer-team order–transport correlations:")
    print(pd.DataFrame(team_stats).set_index("team").round(3))
else:
    print("No 'team' column in centroid_order_runs.")

## 4. Speed Confounding

Is the order effect on duration/length driven purely by the fact that
high-order frames move faster?  We test this with partial correlations.

- Raw $r(p_{\text{mean}},\, T)$: correlation without controlling for speed.
- Partial $r(p_{\text{mean}},\, T \mid \bar v)$: residual correlation after removing the linear effect of speed.

In [ ]:
def partial_corr(df: pd.DataFrame, x: str, y: str, z: str) -> float:
    """Partial correlation of x and y controlling for z."""
    r_xy = df[x].corr(df[y])
    r_xz = df[x].corr(df[z])
    r_yz = df[y].corr(df[z])
    denom = np.sqrt((1 - r_xz**2) * (1 - r_yz**2))
    return np.nan if abs(denom) < 1e-9 else (r_xy - r_xz * r_yz) / denom


df_c = centroid_runs[["duration_s", "run_length_m", "v_mean_mps", "p_mean"]].dropna()

r_pd,   pval_pd = stats.pearsonr(df_c["p_mean"], df_c["duration_s"])
r_pl,   pval_pl = stats.pearsonr(df_c["p_mean"], df_c["run_length_m"])
r_pv,   _       = stats.pearsonr(df_c["p_mean"], df_c["v_mean_mps"])

pr_pd = partial_corr(df_c, "p_mean", "duration_s",   "v_mean_mps")
pr_pl = partial_corr(df_c, "p_mean", "run_length_m", "v_mean_mps")

summary_conf = pd.DataFrame([
    {"outcome": "duration", "raw_r": r_pd, "raw_p": pval_pd, "partial_r_given_speed": pr_pd},
    {"outcome": "length",   "raw_r": r_pl, "raw_p": pval_pl, "partial_r_given_speed": pr_pl},
    {"outcome": "speed (order~speed)", "raw_r": r_pv, "raw_p": np.nan, "partial_r_given_speed": np.nan},
])
print(summary_conf.set_index("outcome").round(4))
print()
if abs(pr_pd) > 0.1:
    print("Order effect on duration persists after controlling for speed — not purely speed-mediated.")
else:
    print("Partial correlation is small — order effect on duration may be largely speed-mediated.")

In [ ]:
fig_conf, axes_conf = plt.subplots(1, 2, figsize=(9, 4))

sc = axes_conf[0].scatter(df_c["p_mean"], df_c["duration_s"],
                           c=df_c["v_mean_mps"], cmap="YlOrRd", s=8, alpha=0.5)
plt.colorbar(sc, ax=axes_conf[0], label="Speed (m/s)")
axes_conf[0].set_xlabel("Polarisation $p$")
axes_conf[0].set_ylabel("Duration (s)")
axes_conf[0].set_title(f"Order vs Duration (raw $r$={r_pd:.2f}, partial={pr_pd:.2f})")

sc2 = axes_conf[1].scatter(df_c["p_mean"], df_c["run_length_m"],
                            c=df_c["v_mean_mps"], cmap="YlOrRd", s=8, alpha=0.5)
plt.colorbar(sc2, ax=axes_conf[1], label="Speed (m/s)")
axes_conf[1].set_xlabel("Polarisation $p$")
axes_conf[1].set_ylabel("Length (m)")
axes_conf[1].set_title(f"Order vs Length (raw $r$={r_pl:.2f}, partial={pr_pl:.2f})")

plt.tight_layout()
save_figure(fig_conf, "supp_robustness_speed_confounding")
plt.show()

## 5. Selection Effect vs True Age Dependence

**Outstanding question**: Is the inverse-age hazard real (runs genuinely stabilise
with age) or a selection artefact (only high-order runs survive to old age,
and high-order runs have lower baseline hazard)?

**Test**: Among runs that have *already survived* to age ≥ 10 s, split by their
order state at run **start** ($p_{\text{start}}$, not the retrospective mean).
If hazard is still lower for high-order survivors, there is a true protective effect.

In [ ]:
haz_ivls = load_cache("hazard_intervals")

# Compute age_bin (2s bins) from age_mid_s
BIN_W = 2.0
haz_ivls = haz_ivls.copy()
haz_ivls["age_bin"] = np.floor(haz_ivls["age_mid_s"] / BIN_W) * BIN_W + BIN_W / 2

# Runs that survived to at least 10 seconds
long_run_uids = haz_ivls.loc[haz_ivls["age_bin"] >= 10, "run_uid"].unique()
haz_long = haz_ivls.loc[haz_ivls["run_uid"].isin(long_run_uids)].copy()
print(f"Runs surviving to age >= 10s: {len(long_run_uids):,} of {haz_ivls['run_uid'].nunique():,}")

# Join p_start from centroid_runs via run_uid
p_start_map = centroid_runs.set_index("run_uid")["p_start"]
haz_long["p_start"] = haz_long["run_uid"].map(p_start_map)

if haz_long["p_start"].notna().any():
    p_start_med = haz_long.groupby("run_uid")["p_start"].first().median()

    run_p_start = haz_long.groupby("run_uid")["p_start"].first()
    haz_long["start_state"] = haz_long["run_uid"].map(
        run_p_start.apply(lambda p: "hi_start" if p >= p_start_med else "lo_start")
    )

    fig_sel, ax_sel = plt.subplots(figsize=(5.5, 3.5))
    for group, color, ls in [
        ("lo_start", "#4393c3", "--"),
        ("hi_start", "#d6604d", "-"),
    ]:
        sub = haz_long.loc[haz_long["start_state"] == group]
        h_s = (
            sub.groupby("age_bin")["event"]
            .agg(events="sum", at_risk="count")
            .assign(h=lambda d: d["events"] / d["at_risk"])
            .reset_index()
        )
        n_runs = sub["run_uid"].nunique()
        ax_sel.plot(h_s["age_bin"].values.astype(float),
                    h_s["h"].values, "o-",
                    color=color, ls=ls, lw=1.2, ms=4,
                    label=f"{group}  (N={n_runs:,})")

    ax_sel.set_xlabel("Age (s)")
    ax_sel.set_ylabel("Hazard $h(a)$")
    ax_sel.set_title("Hazard among survivors, conditioned on run-start order")
    ax_sel.legend(fontsize=8)
    plt.tight_layout()
    save_figure(fig_sel, "supp_robustness_selection_effect")
    plt.show()
else:
    print("p_start could not be joined — check run_uid alignment between hazard_intervals and centroid_runs.")

## Summary Table

In [ ]:
summary = pd.DataFrame([
    {
        "Check": "theta=20 sensitivity",
        "Status": "cached" if 20 in theta_results and len(theta_results[20]) > 0 else "needs cache",
        "Finding": "TBD" if 20 not in theta_results else f"median_dur={theta_results[20]['duration_s'].median():.1f}s",
    },
    {
        "Check": "theta=30 (base)",
        "Status": "complete",
        "Finding": f"median_dur={centroid_runs['duration_s'].median():.1f}s",
    },
    {
        "Check": "theta=40 sensitivity",
        "Status": "cached" if 40 in theta_results and len(theta_results[40]) > 0 else "needs cache",
        "Finding": "TBD" if 40 not in theta_results else f"median_dur={theta_results[40]['duration_s'].median():.1f}s",
    },
    {
        "Check": "Speed confounding (partial r: order~duration|speed)",
        "Status": "complete",
        "Finding": f"raw r={r_pd:.3f}, partial r={pr_pd:.3f}",
    },
    {
        "Check": "Speed confounding (partial r: order~length|speed)",
        "Status": "complete",
        "Finding": f"raw r={r_pl:.3f}, partial r={pr_pl:.3f}",
    },
])
print(summary.to_string(index=False))